# Edge IIoT - Binary Classification


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.10.0+cu128
True
12.8
1


In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")


--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/processed/edge_iiot

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/processed/edge_iiot/ML-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (150005, 59)


## 1. Undersampling and Balancing

In [5]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [6]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [7]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24231
DDoS_UDP                 14498
DDoS_ICMP                13306
DDoS_HTTP                10560
SQL_injection            10297
Uploading                10260
Ransomware               10196
Vulnerability_scanner    10065
Backdoor                  9992
Password                  9980
Port_Scanning             9690
XSS                       9620
DDoS_TCP                  6011
Fingerprinting             932
MITM                       367
Name: count, dtype: int64

TOTAL:  150005


In [8]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24231           | 3000           
DDoS_UDP                  | 14498           | 3000           
DDoS_ICMP                 | 13306           | 3000           
DDoS_HTTP                 | 10560           | 3000           
SQL_injection             | 10297           | 3000           
Uploading                 | 10260           | 3000           
Ransomware                | 10196           | 3000           
Vulnerability_scanner     | 10065           | 3000           
Backdoor                  | 9992            | 3000           
Password                  | 9980            | 3000           
Port_Scanning             | 9690            | 3000           
XSS                       | 9620            | 3000           
DDoS_TCP                  | 6011            | 3000           
Fingerprinting            | 932             | 932            
MITM          

In [9]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(40299, 58)


In [10]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [11]:
# Select only numeric
print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.request.uri.query', 'http.referer',
       'http.request.version', 'mqtt.msg', 'proto', 'http.request.path'],
      dtype='str')


Remaining categorical cols: 0


In [12]:
# Remove env-specific columns
cols_to_drop = [
    'frame.time.delta', 'frame.time.order',
    *[c for c in X.columns if c.startswith('ip.src_category') or c.startswith('ip.dst_category')]
]
X = X.drop(columns=cols_to_drop, errors='ignore')

In [13]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [14]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (32239, 40)
X_test shape: (8060, 40)


In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [16]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/33 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [17]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
CatBoostClassifier,0.989206,0.928266,0.993119,0.988825,0.989306,0.989206,0.986259,0.001406,0.909049,0.010268,0.991378,0.000918,0.985624,0.001557,0.986411,0.001338,0.986259,0.001406,14.380105
LGBMClassifier,0.989206,0.927500,0.993061,0.988815,0.989330,0.989206,0.987810,0.000776,0.918508,0.005391,0.991238,0.001013,0.987309,0.000846,0.987955,0.000750,0.987810,0.000776,662.909721
XGBClassifier,0.988958,0.926600,0.992375,0.988558,0.989063,0.988958,0.987531,0.000738,0.918357,0.005142,0.991716,0.001063,0.987032,0.000805,0.987620,0.000719,0.987531,0.000738,3.227731
KNeighborsClassifier,0.988586,0.923333,0.927428,0.988147,0.988725,0.988586,0.987158,0.000660,0.914325,0.004783,0.915342,0.004395,0.986603,0.000726,0.987313,0.000631,0.987158,0.000660,2.962908
ExtraTreesClassifier,0.985360,0.901667,0.992993,0.984619,0.985588,0.985360,0.984615,0.002081,0.897050,0.014137,0.991394,0.001148,0.983780,0.002302,0.984853,0.002017,0.984615,0.002081,1.479982
BaggingClassifier,0.985360,0.901667,0.985401,0.984619,0.985588,0.985360,0.984398,0.001443,0.895208,0.009696,0.983956,0.002165,0.983541,0.001605,0.984659,0.001397,0.984398,0.001443,2.162947
ExtraTreeClassifier,0.985360,0.901667,0.992132,0.984619,0.985588,0.985360,0.984212,0.001504,0.894725,0.009938,0.988466,0.001718,0.983348,0.001669,0.984439,0.001467,0.984212,0.001504,1.588810
DecisionTreeClassifier,0.985236,0.901600,0.985368,0.984496,0.985432,0.985236,0.984398,0.001402,0.895400,0.009486,0.983472,0.002328,0.983545,0.001560,0.984649,0.001355,0.984398,0.001402,1.811412
RandomForestClassifier,0.985236,0.900833,0.985421,0.984481,0.985468,0.985236,0.984274,0.001486,0.894375,0.009983,0.985753,0.003015,0.983402,0.001653,0.984539,0.001438,0.984274,0.001486,3.463760


In [18]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(5))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.989206,0.927500,0.993061,0.988815,0.989330,0.989206,0.987810,0.000776,0.918508,0.005391,0.991238,0.001013,0.987309,0.000846,0.987955,0.000750,0.987810,0.000776,662.909721
XGBClassifier,0.988958,0.926600,0.992375,0.988558,0.989063,0.988958,0.987531,0.000738,0.918357,0.005142,0.991716,0.001063,0.987032,0.000805,0.987620,0.000719,0.987531,0.000738,3.227731
KNeighborsClassifier,0.988586,0.923333,0.927428,0.988147,0.988725,0.988586,0.987158,0.000660,0.914325,0.004783,0.915342,0.004395,0.986603,0.000726,0.987313,0.000631,0.987158,0.000660,2.962908
CatBoostClassifier,0.989206,0.928266,0.993119,0.988825,0.989306,0.989206,0.986259,0.001406,0.909049,0.010268,0.991378,0.000918,0.985624,0.001557,0.986411,0.001338,0.986259,0.001406,14.380105
ExtraTreesClassifier,0.985360,0.901667,0.992993,0.984619,0.985588,0.985360,0.984615,0.002081,0.897050,0.014137,0.991394,0.001148,0.983780,0.002302,0.984853,0.002017,0.984615,0.002081,1.479982
